In [1]:
import csv
from datetime import datetime
import logging 

logging.basicConfig(level=logging.INFO)

In [2]:
# extract: stream data row-by-row

def extract_data(file_path):
    logging.info("Extracting file")
    with open(file_path, mode="r", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        for row in reader:
            yield row

In [5]:
# transform: process one record at a time
def transform_data(records):
    logging.info("Transforming records")
    curr_time = datetime.now()
    for row in records:
        row = {
            "transaction_id": row.get("transaction_id"),
            "customer_id": row.get("customer_id"),
            "customer_name": row.get("customer_name"),
            "transaction_date": row.get("transaction_date"),
            "price": float(row.get("amount")),
            "currency": row.get("currency"),
            "is_usd": "True" if row.get("currency") == "USD" else "False",
            "product": row.get("product"),
            "processing_time": datetime.strftime(curr_time, "%Y-%m-%dT%H:%M:%S")
        }

        yield row

In [12]:
# load: write chunks directly to outfile
def load_data(transformed_records, outfile):
    logging.info(f"Loading data to {outfile}")
    first_row = next(transformed_records, None)
    field_names = first_row.keys()

    with open(outfile, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames = field_names)
        writer.writeheader()
        writer.writerow(first_row)

        for row in transformed_records:
            writer.writerow(row)

In [13]:
raw_stream = extract_data("../sample_data/sample_transactions.csv")
cleaned_stream = transform_data(raw_stream)
load_data(cleaned_stream, "../output/cleaned_transactions.csv")

INFO:root:Loading data to ../output/cleaned_transactions.csv
INFO:root:Transforming records
INFO:root:Extracting file
